# LLM-RecFusion — Stage 5: 装载高吞吐引擎——vLLM 推理框架部署

**Objective**: 淘汰 HuggingFace 原生低效的 `generate()`，使用 vLLM 框架加载 AWQ 量化模型，实现高并发的批量推理测试，满足推荐系统对 P99 延迟的极高要求。

---

### 部署架构

```
┌─────────────────────────────────────────────────────────────────────────────┐
│ 任务二产出：AWQ INT4 量化模型 (显存 ~0.9GB)                                  │
└──────────────────────────┬──────────────────────────────────────────────────┘
                           │
                           ▼
┌─────────────────────────────────────────────────────────────────────────────┐
│ 任务三：vLLM 推理引擎 🚀                                                    │
│                                                                             │
│  ┌──────────────────────────────────────────────────────────────────┐       │
│  │ PagedAttention：KV Cache 分页化管理，碎片率 < 4%                   │       │
│  │ Continuous Batching：请求级动态调度，GPU 利用率拉满                │       │
│  └──────────────────────────────────────────────────────────────────┘       │
└──────────────────────────┬──────────────────────────────────────────────────┘
                           │
                           ▼
┌─────────────────────────────────────────────────────────────────────────────┐
│ 生产 API：1 次前向传播 → Top-5 重排结果 < 500ms P99                        │
└─────────────────────────────────────────────────────────────────────────────┘
```

---

## 1. 环境与依赖准备

导入 `vllm` 核心库。如果当前环境缺少 GPU 或 vLLM 未安装，Notebook 仍能优雅降级完成文字预览。

In [ ]:
# ============================================================
# 1. 环境与依赖准备
# ============================================================

import sys
import os
import re
import json
import time
import warnings
from typing import List, Optional, Dict, Any

warnings.filterwarnings('ignore')

# ---------- vLLM 推理引擎 ----------
# vLLM: 高性能 LLM 推理框架，核心是 PagedAttention 和 Continuous Batching
# 是线上部署 LLM 推荐重排的事实标准
try:
    from vllm import LLM, SamplingParams
    VLLM_AVAILABLE = True
    print("[✓] vLLM 导入成功")
except ImportError:
    VLLM_AVAILABLE = False
    print("[!] vLLM 未安装。推理核心功能不可用，但本 Notebook 仍会展示代码模板。")
    print("    安装命令: pip install vllm")

# ---------- 可选：检查 GPU 可用性 ----------
try:
    import torch
    if torch.cuda.is_available():
        gpu_name = torch.cuda.get_device_name(0)
        gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1024**3
        print(f"[✓] GPU 可用: {gpu_name} ({gpu_mem:.1f} GB VRAM)")
    else:
        print("[!] GPU 不可用，vLLM 需要 GPU 才能运行")
except ImportError:
    print("[!] PyTorch 未安装，跳过 GPU 检测")

print("\n环境就绪，开始构建 vLLM 推理引擎...")

## 2. Prompt 构造器（复用任务一成果）

从任务一的 Notebook 中移植核心函数，构建让 LLM 对 `[1,2,3,4,5]` 进行重排的 Prompt。

In [ ]:
# ============================================================
# 2a. 模拟用户画像与候选物品（复用任务一对抗性测试设计）
# ============================================================

# ---------- 模拟用户画像 ----------
user_profile: str = (
    "该用户近期频繁浏览和购买：高配游戏本、机械键盘、人体工学椅、"
    "高刷新率电竞显示器、游戏鼠标。用户偏好明确：高性能电竞装备，"
    "客单价 2000~15000 元，对品牌和性能敏感，对非电竞类日用品不感兴趣。"
)

# ---------- 构造对抗性候选列表 ----------
# 故意混入 2 个不相关爆款，测试 LLM 重排能力
candidate_list: List[Dict[str, Any]] = [
    {
        "id": 1,
        "name": "ROG 魔霸 7 Plus 高配游戏本",
        "category": "电脑/游戏本",
        "price": 12999,
        "desc": "RTX4070 + R9 7845HX + 17.3英寸 2.5K 240Hz 电竞屏"
    },
    {
        "id": 2,
        "name": "赫莲娜 黑白绷带面霜套装",
        "category": "美妆/护肤",
        "price": 3880,
        "desc": "玻色因修护抗老，贵妇级面霜，月销 10 万+ 爆款"
    },
    {
        "id": 3,
        "name": "罗技 G Pro X Superlight 无线游戏鼠标",
        "category": "外设/鼠标",
        "price": 899,
        "desc": "63g 超轻量、HERO 25K 传感器、专业电竞级无线鼠标"
    },
    {
        "id": 4,
        "name": "爱他美 卓傲 3段 幼儿奶粉",
        "category": "母婴/奶粉",
        "price": 298,
        "desc": "德国原装进口，益生元配方，1-3岁幼儿适用，月销 20 万+"
    },
    {
        "id": 5,
        "name": "Ergoup 有谱 蝴蝶人体工学椅",
        "category": "家居/电脑椅",
        "price": 3599,
        "desc": "4D 扶手 + 135° 后仰 + 全网面透气，程序员/电竞玩家首选"
    },
]

print(f"候选物品数: {len(candidate_list)}")
for item in candidate_list:
    print(f"  [{item['id']}] {item['name']}  ¥{item['price']}")

In [ ]:
# ============================================================
# 2b. 核心 Prompt 构造器（移植自 Notebook 15）
# ============================================================


def build_listwise_prompt(
    user_profile: str,
    candidate_list: List[Dict[str, Any]],
) -> str:
    """
    构建 Listwise 重排 Prompt。

    三层次模板设计：
      1. 角色设定 → 明确 LLM 的推荐专家身份
      2. 上下文注入 → 用户画像 + 候选物品列表
      3. 输出格式强制约束 → 防幻觉、防话痨

    Parameters
    ----------
    user_profile : str
        用户兴趣画像描述文本。
    candidate_list : List[Dict[str, Any]]
        候选物品列表，每个元素需包含 id, name, category, price, desc。

    Returns
    -------
    str
        构造完成的 Prompt 字符串。
    """
    # ---------- 格式化候选物品列表 ----------
    candidate_lines: List[str] = []
    for item in candidate_list:
        candidate_lines.append(
            f"[{item['id']}] {item['name']}\n"
            f"    品类: {item['category']} | 价格: ¥{item['price']}\n"
            f"    描述: {item['desc']}"
        )
    candidates_text: str = "\n".join(candidate_lines)

    # ---------- 三层次 Prompt 模板 ----------
    prompt: str = f"""\
## 角色设定
你是一个大厂资深电商推荐专家，请站在全局视角对以下推荐列表进行重排，以最大化用户的点击兴趣和列表多样性。

## 上下文注入
### 用户画像
{user_profile}

### 候选商品列表（输入）
{candidates_text}

## 输出格式强制约束
请严格按以下要求输出：
1. 禁止输出任何多余的解释、引言、评论或过渡语（例如"好的"、"我已为您排好序"等）。
2. **仅输出** 一个 Python 风格的数字列表，表示按推荐程度从高到低排列的序号。
3. 格式必须严格遵循：[3, 1, 4, 2, 5]
4. 序号范围必须在 [1, 2, 3, 4, 5] 内，且不允许重复和遗漏。

输出："""

    return prompt


# ---------- 构建单条 Prompt ----------
prompt: str = build_listwise_prompt(user_profile, candidate_list)

print("=" * 70)
print("构建完成的 Prompt（单条）:")
print("=" * 70)
print(prompt)
print("=" * 70)
print(f"\nPrompt 总字符数: {len(prompt)}")

## 3. vLLM 引擎初始化

配置 vLLM 引擎参数，加载 AWQ 量化模型。

### 显存优化参数详解

| 参数 | 值 | 工程意义 |
|------|-----|---------|
| `gpu_memory_utilization` | 0.8 | 避免显存爆裂，留出 20% 余量给模型权重加载和推理临时缓冲区 |
| `max_model_len` | 1024 | 推荐重排是短文本任务，不需要 2048+ 上下文，截断上下文可极限压榨 KV Cache 显存 |
| `enforce_eager` | True | 跳过 CUDA 图优化（建图失败时降级），确保在低端显卡上稳定初始化 |

> **显存算账**：以 Qwen-1.8B INT4 为例，~0.9GB 权重 + KV Cache(1024 ctx × 16 layers × 2 heads × 2 for K+V × 4 bytes ≈ 0.5GB) ≈ 1.5GB，配合 80% 利用率，6GB 显卡绰绰有余。

In [ ]:
# ============================================================
# 3. vLLM 引擎初始化
# ============================================================

# ---------- 模型路径配置 ----------
# 生产环境请替换为任务二 AWQ 量化产出的实际模型路径
# 例如: MODEL_PATH = "./models/qwen1.5-1.8b-chat-awq"
MODEL_PATH: str = "Qwen/Qwen1.5-1.8B-Chat-AWQ"

if VLLM_AVAILABLE:
    try:
        # ============================================================
        # 引擎参数配置（面试高频考点）
        # ============================================================

        # [工程意义]
        # gpu_memory_utilization=0.8:
        #   vLLM 默认会尝试占满所有可用显存（=1.0），但在消费级显卡上
        #   极易触发 OOM。设置为 0.8 意味着 vLLM 只会使用 80% 的显存，
        #   剩余 20% 留给模型权重加载时的临时缓冲区、推理过程中的中间
        #   激活值等。这是一个"宁可降速也不爆显存"的生产级保守策略。
        #   在 6GB 显卡上，可用显存 = 6 * 0.8 = 4.8GB，对 1.8B INT4
        #   （~0.9GB 权重）绰绰有余。
        #
        # max_model_len=1024:
        #   vLLM 的 KV Cache 显存预分配量与 max_model_len 成正比。
        #   推荐重排是短文本任务（Prompt ~500 tokens），不需要 2048+
        #   的超长上下文。将 max_model_len 从 2048 截断到 1024，KV Cache
        #   显存占用直接减半，这对显存受限设备至关重要。
        #
        # enforce_eager=True:
        #   vLLM 默认使用 CUDA 图优化来加速推理，但 CUDA 图构建在某些
        #   显卡/驱动组合上可能失败。启用 enforce_eager 跳过 CUDA 图，
        #   虽然略有性能损失（~5-10%），但能保证引擎稳定初始化。
        #   线上生产环境如果 CUDA 图稳定，建议关闭该选项（设为 False）。

        llm = LLM(
            model=MODEL_PATH,
            quantization="awq",          # 显式声明使用 AWQ INT4 解码
            gpu_memory_utilization=0.8,   # 显存占用上限 80%，防爆显存
            max_model_len=1024,            # 最大上下文 1024 tokens，节省 KV Cache
            enforce_eager=True,            # 跳过 CUDA 图，保证低端显卡稳定初始化
        )
        print("[✓] vLLM 引擎初始化成功！")
        print(f"    模型: {MODEL_PATH}")
        print(f"    量化: AWQ")
        print(f"    显存利用率: 0.8")
        print(f"    最大上下文: 1024 tokens")

    except Exception as e:
        error_msg = str(e).lower()
        # ----- OOM 降级处理 -----
        if "out of memory" in error_msg or "cuda out of memory" in error_msg:
            print("[!] OOM 错误：显存不足以加载模型。")
            print("    降级建议:")
            print("      1. 降低 gpu_memory_utilization=0.6 进一步限制显存")
            print("      2. 降低 max_model_len=512 进一步压缩 KV Cache")
            print("      3. 尝试更小的量化模型（如 Qwen2.5-0.5B-Instruct-AWQ）")
            print("      4. 关闭其他占用显存的进程（nvidia-smi 查看）")
            llm = None
        else:
            # ----- 其他异常降级 -----
            print(f"[!] vLLM 引擎初始化失败: {e}")
            print("    降级建议:")
            print("      1. 确认 vLLM 版本兼容: pip install --upgrade vllm")
            print("      2. 确认模型路径正确: 检查 MODEL_PATH")
            print("      3. 如果 CUDA 图报错，确保 enforce_eager=True")
            llm = None
else:
    print("[!] vLLM 不可用，跳过引擎初始化")
    llm = None

print("\n引擎状态:", "[✓] 就绪" if llm is not None else "[✗] 未加载")

## 4. 批量推理与采样参数配置

### 采样策略：重排是判别任务，不是生成任务

与 Chat 场景不同，推荐重排需要的是**确定性排序**，而非创造性生成。因此采样参数必须极端保守：
- `temperature=0.0`：完全贪婪解码，禁止任何随机性
- `top_p=1.0`：不截断词汇概率分布
- `stop` 词：防止模型在输出完数组后继续废话（复用任务一测试的 stop 词）

In [ ]:
# ============================================================
# 4. 构造重排请求与采样参数
# ============================================================

# ---------- 并发 Mock 数据 ----------
# 使用列表推导式生成 10 个完全相同的重排 Prompt
# 模拟线上同时有 10 个用户发起重排请求
BATCH_SIZE: int = 10
prompts: List[str] = [prompt] * BATCH_SIZE

print(f"并发请求数: {BATCH_SIZE}")
print(f"每个 Prompt 长度: {len(prompt)} chars")
print(f"批量推理总字符数: {len(prompt) * BATCH_SIZE} chars")

# ---------- 采样参数配置 ----------
# 重排 = 判别性任务，严禁模型发散"幻觉"
# temperature=0.0 → 贪婪解码，保证每次输出一致
# top_p=1.0       → 不截断概率分布，允许所有词汇参与
# stop 词         → 防止模型在输出完数组后继续废话

sampling_params = SamplingParams(
    temperature=0.0,    # 贪婪解码，判别任务不需要随机性
    top_p=1.0,          # 不截断，保证完整的概率空间
    max_tokens=256,     # 限制最大生成长度，避免模型无限发散
    stop=["\n\n", "Output:", "输出："],  # 防废话 stop 词
)

print("\n采样参数:")
print(f"  temperature = {sampling_params.temperature}")
print(f"  top_p       = {sampling_params.top_p}")
print(f"  max_tokens  = {sampling_params.max_tokens}")
print(f"  stop        = {sampling_params.stop}")
print("\n等待批量推理...")

## 5. 批量推理与极速 Benchmark

调用 `llm.generate()` 进行批量推理，使用 `time` 模块精确统计 10 个请求并发处理的总耗时。

In [ ]:
# ============================================================
# 5. 批量推理与极速 Benchmark
# ============================================================

if llm is not None:
    try:
        # ---------- 计时开始 ----------
        start_time: float = time.perf_counter()

        # ---------- 批量推理 ----------
        # llm.generate() 内部自动进行 Continuous Batching
        # vLLM 将 BATCH_SIZE 个请求的 KV Cache 以 Block 为单位动态调度
        # 真正实现了请求级别的并行处理
        outputs = llm.generate(prompts, sampling_params)

        # ---------- 计时结束 ----------
        elapsed: float = time.perf_counter() - start_time

        print("=" * 70)
        print(f"批量推理完成！总耗时: {elapsed:.3f} 秒 ({elapsed*1000:.1f} ms)")
        print(f"平均每请求: {elapsed / BATCH_SIZE * 1000:.2f} ms")
        print(f"吞吐量: {BATCH_SIZE / elapsed:.1f} req/s")
        print("=" * 70)

    except Exception as e:
        error_msg = str(e).lower()
        if "out of memory" in error_msg or "cuda out of memory" in error_msg:
            print("[!] 批量推理 OOM！")
            print("    降级建议:")
            print("      1. 减小 BATCH_SIZE（如从 10 降到 4）")
            print("      2. 进一步降低 max_model_len")
            print("      3. 减少 gpu_memory_utilization 为更小模型预留显存")
        else:
            print(f"[!] 批量推理失败: {e}")
else:
    print("[!] vLLM 引擎未加载，跳过批量推理。")
    print("    下方的输出解析和结论部分仍可正常展示。")
    outputs = None
    elapsed = 0.0

## 6. 输出解析与结果验证

复用任务一中开发的鲁棒正则解析器，从模型输出中提取排序数组，验证每个输出的准确性和合法性。

In [ ]:
# ============================================================
# 6. 输出解析与结果验证
# ============================================================

# ---------- 鲁棒正则解析器（移植自 Notebook 15） ----------

def parse_llm_output(text: str) -> Optional[List[int]]:
    """
    从 LLM 输出中鲁棒地提取排序序号数组。

    无论大模型是否输出了废话，只要文本中包含类似 [x, x, x, x, x]
    的结构，就能将其安全地解析为 Python List。
    """
    if not text or not text.strip():
        return None

    # 核心正则：匹配方括号内逗号分隔的数字
    pattern = r"\[\s*(\d+(?:\s*,\s*\d+)*)\s*\]"
    matches: List[str] = re.findall(pattern, text)

    if not matches:
        return None

    # 取最后一个匹配（跳过模型中间推理过程产生的数组）
    digits_str: str = matches[-1]
    parsed: List[int] = [int(x.strip()) for x in digits_str.split(",")]

    return parsed


def validate_rerank_indices(
    indices: Optional[List[int]],
    expected_len: int = 5,
) -> bool:
    """
    校验重排结果是否合法。

    检查项：
      - 不能为 None
      - 长度与候选列表一致
      - 包含所有 1..N 序号（无重复、无遗漏）
    """
    if indices is None:
        return False
    if len(indices) != expected_len:
        return False
    if sorted(indices) != list(range(1, expected_len + 1)):
        return False
    return True


# ---------- 解析并打印批量输出 ----------
if outputs is not None:
    print("=" * 70)
    print(f"批量推理结果 (BATCH_SIZE={BATCH_SIZE})")
    print("=" * 70)

    valid_count: int = 0
    invalid_count: int = 0

    for idx, output in enumerate(outputs):
        # 提取生成的文本
        generated_text: str = output.outputs[0].text
        # 计算生成的 token 数
        generated_tokens: int = len(output.outputs[0].token_ids)

        # 解析排序数组
        parsed: Optional[List[int]] = parse_llm_output(generated_text)
        is_valid: bool = validate_rerank_indices(parsed, expected_len=5)

        if is_valid:
            valid_count += 1
            status = "✓"
        else:
            invalid_count += 1
            status = "✗"

        print(f"\n--- 请求 {idx + 1}/{BATCH_SIZE} [{status}] ---")
        print(f"  原始输出: {generated_text!r}")
        print(f"  生成 tokens: {generated_tokens}")
        print(f"  解析结果: {parsed}")

        if is_valid:
            print(f"  重排顺序:")
            for rank, item_id in enumerate(parsed, 1):
                item = candidate_list[item_id - 1]
                print(f"    Top-{rank}: [{item['id']}] {item['name']}  ¥{item['price']}")

    print("=" * 70)
    print(f"汇总: {valid_count}/{BATCH_SIZE} 通过校验, {invalid_count}/{BATCH_SIZE} 未通过")
    print(f"总耗时: {elapsed:.3f}s, 平均每请求: {elapsed / BATCH_SIZE * 1000:.1f}ms")
    print("=" * 70)

else:
    print("[!] 无输出数据，跳过解析。")
    print("    （如果 vLLM 引擎未加载这是正常现象）")

## 7. 极客硬核笔记：为什么一定要上 vLLM？（面试高光素材）

---

### 核心论点：HuggingFace 原生推理的显存灾难

原生 HuggingFace `generate()` 在处理 Batch 请求时，存在一个被严重低估的性能杀手——**KV Cache 的连续预分配机制**。

#### 问题根源

```
HF 原生:  KV Cache 连续存储 (预先分配最大长度)
┌──────────────────────────────────────────────────────┐
│  [Seq 1] ████████████████████░░░░░░░░░░░░░░░░░░░░  │
│  [Seq 2] ██████░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░  │
│  [Seq 3] ████████████████████████████░░░░░░░░░░░░  │
│                                                  │
│  每个序列都按 max_model_len 预分配连续显存         │
│  实际使用空间: ██  未使用但被占用的空间: ░░        │
│  显存碎片率: 60%+                                │
└──────────────────────────────────────────────────────┘
"""

当 Batch 中的多个序列生成长度不一致时（实际场景必然如此）：
1. 每个序列的 KV Cache 都按 `max_model_len` 预分配了最大长度的连续显存
2. 较短的序列生成完毕后，其未使用的显存无法被其他序列复用（因为是连续分配）
3. 结果是：**高达 60% 以上的显存被白白浪费**，并发量被死死卡住

#### vLLM 的解决方案：PagedAttention

```
vLLM: PagedAttention — KV Cache 分页化管理
┌──────────────────────────────────────┐
│  物理显存空间 (Block = 16 tokens)    │
│  ┌──────┐┌──────┐┌──────┐┌──────┐  │
│  │ S1-A  ││ S2-A  ││ S1-B  ││ S3-A  │  ← 非连续存储
│  ├──────┤├──────┤├──────┤├──────┤  │
│  │ S3-B  ││ S2-B  ││ Free  ││ Free  │  ← 空闲块可被任何序列复用
│  └──────┘└──────┘└──────┘└──────┘  │
│                                     │
│  逻辑视图:                           │
│  S1: [Block A] → [Block B]          │
│  S2: [Block A] → [Block B]          │
│  S3: [Block A] → [Block B]          │
│                                     │
│  显存碎片率: < 4%                   │
└──────────────────────────────────────┘
"""

PagedAttention 的核心突破：

1. **分页管理**：将 KV Cache 划分为固定大小的 Block（典型值 16 tokens），以 Block 为单位分配显存
2. **非连续存储**：一个序列的 KV Cache 可以分散存储在物理显存的任意位置，通过页表映射索引
3. **按需分配**：序列生成多少 token，就分配多少 Block，绝不浪费
4. **块级复用**：已完成的序列释放的 Block 可以被任意新序列立即复用

#### 量化对比

| 维度 | HuggingFace 原生推理 | vLLM (PagedAttention) |
|------|---------------------|-----------------------|
| KV Cache 分配方式 | 连续预分配 | 非连续按需分配 |
| 显存碎片率 | 60%+ | < 4% |
| Batch 调度 | 静态 Batch | Continuous Batching |
| 批量推理吞吐量 | 基线 (1x) | 8-23x 提升 |
| P99 延迟抖动 | 高（OOM 风险） | 低（稳定） |
| 对 6GB 显卡友好度 | ❌ 极易 OOM | ✅ 从容运行 |

### 为什么这对推荐系统至关重要？

推荐系统的 P99 延迟要求通常 < 500ms。如果使用 HuggingFace 原生推理：
- 单请求推理可能只要 100ms，但一旦并发上升到 10+，OOM 挂掉或触发显存交换，P99 直接飙升到 5s+
- 为了防 OOM 被迫缩小 Batch Size，吞吐量大幅下降

而 vLLM + AWQ 的组合拳：
- AWQ INT4 量化将模型权重从 ~3.6GB 压缩到 ~0.9GB（任务二完成）
- PagedAttention 将 KV Cache 显存利用率从 40% 提升到 96%+
- Continuous Batching 实现请求级动态调度，GPU 利用率始终饱和
- **最终结果：单张 6GB 消费级显卡，实现 10+ 并发毫秒级重排响应**

### 面试速记卡

```
Q: 为什么 vLLM 比 HuggingFace generate() 快那么多？

A: 三个核心原因：
  1. PagedAttention — 显存碎片率从 60%+ 降至 < 4%，同样的显存能装更多请求
  2. Continuous Batching — 请求级动态调度，GPU 永不空闲
  3. 极致 Kernel 优化 — 融合了 FlashAttention、量化 kernel 等底层加速

Q: PagedAttention 和操作系统的虚拟内存有什么关系？

A: 本质一模一样！OS 的虚拟内存用页表将逻辑地址映射到物理内存页，
   解决物理内存碎片化和进程隔离问题。PagedAttention 就是将这个
   经典思想应用到 KV Cache 管理上，用 Block Table 做逻辑 Block →
   物理 Block 的映射。这就是"计算系统设计中的分治思想"。
"""

---

*LLM-RecFusion — AWQ 量化榨干权重显存 + PagedAttention 榨干 KV Cache 显存 = 消费级显卡上的 LLM 推荐重排部署方案。*